# Grip Strength-to-Weight Ratio (GSWR) and Incident Stroke in Chinese Middle-Aged and Older Adults
## CHARLS 2011-2020: Complete Analysis Pipeline

**Analysis notebook accompanying the manuscript:**  
*Grip Strength-to-Weight Ratio and Incident Stroke in Chinese Middle-Aged and Older Adults: A Prospective Cohort Study with Mediation Analysis*

---
### Analysis Overview
1. Setup & Data Loading
2. Variable Preparation
3. Table 1: Baseline Characteristics by GSWR Quartiles
4. Cox Proportional Hazards Models (5 sequential models)
5. Quartile Analysis & Dose-Response Trends
6. Restricted Cubic Splines (RCS)
7. Mediation Analysis (CRP, Glucose, Hyperglycemia)
8. Interaction Analysis (Multiplicative + Additive)
9. Subgroup Analyses
10. E-value for Unmeasured Confounding
11. Population Attributable Fraction (PAF)
12. Figures
13. Sensitivity Analyses

**Software:** Python 3.8 | lifelines v0.27 | scikit-learn v1.0 | statsmodels v0.13  
**Random seed:** 42  
**Data:** CHARLS publicly available at http://charls.pku.edu.cn

In [ ]:
import pandas as pd
import numpy as np
import os, warnings
warnings.filterwarnings('ignore')

# Survival analysis
from lifelines import CoxPHFitter

# Mediation & interaction
from sklearn.linear_model import LogisticRegression, LinearRegression
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

# Paths
DATA_DIR = r'E:\empowerstats\analysis\claude code skill\academic research skill\data\cleaned'
OUT_DIR = os.path.join(DATA_DIR, '..', 'results')
os.makedirs(OUT_DIR, exist_ok=True)

np.random.seed(42)
print('Libraries loaded. Random seed: 42')

## 1. Load Data & Prepare Variables

In [ ]:
# Load cleaned analysis cohort
df = pd.read_stata(os.path.join(DATA_DIR, 'charls_analysis_cohort.dta'))
print(f'Analysis cohort: {len(df):,} participants')
print(f'Incident strokes: {int(df["stroke_incident"].sum())} ({df["stroke_incident"].mean()*100:.2f}%)')

In [ ]:
# === Outcome ===
df['stroke'] = df['stroke_incident'].astype(int)
df['time']  = np.where(df['stroke'] == 1, 4.0, 9.0)  # approximate survival time

# === Exposure: GSWR ===
df['rhgs'] = df['rhgs'].astype(float)

# === Demographics ===
df['age_5yr']   = df['age'] / 5.0
df['sex_f']     = df['sex'].map({1: 0, 2: 1})          # female=1
df['edu_cat']   = df['education'].map({1: 0, 2: 1, 3: 2})
df['married']   = df['marry_status'].map({1: 1, 2: 0})
df['rural']     = df['living_place'].map({1: 0, 2: 1})

# === Lifestyle ===
df['smoker']    = df['smoke_status'].map({0: 0, 1: 1, 2: np.nan})
df['drinker']   = df['drink_status'].map({0: 0, 1: 1, 2: 2})
df['pa']        = df['phy_act'].astype(float)

# === Clinical ===
df['sbp']       = df['sbp_mean'] / 10.0                  # per 10 mmHg
df['glucose']   = df['glu'] / 10.0                       # per 10 mg/dL
df['tc_mmol']   = df['tc'] / 38.67                       # mg/dL to mmol/L
df['hdl_mmol']  = df['hdl'] / 38.67
df['egfr_10']   = df['egfr'] / 10.0
df['bmi_c']     = df['bmi'].astype(float)
df['htn']       = df['hypertension'].astype(int)
df['dm']        = df['diabetes'].astype(int)
df['dyslip']    = df['dyslipidemia'].astype(int)
df['heart_dz']  = df['dx_heart_disease'].map({1: 1, 0: 0})

# === Biomarkers ===
df['crp_ln']    = np.log(df['crp'] + 0.01)
df['crp_high']  = (df['crp'] > 3).astype(int)

# === Waist (for sensitivity) ===
if 'waist_cm' in df.columns:
    df['waist_clean'] = df['waist_cm'].clip(40, 180)

# === Sex-specific GSWR quartiles ===
df['rhgs_q'] = np.nan
for sex_val in [0, 1]:
    mask = df['sex_f'] == sex_val
    if mask.sum() > 0:
        df.loc[mask, 'rhgs_q'] = pd.qcut(
            df.loc[mask, 'rhgs'], q=4,
            labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop'
        )

print('Variables prepared.')
print(f'  GSWR mean ± SD: {df["rhgs"].mean():.3f} ± {df["rhgs"].std():.3f}')
print(f'  GSWR range: [{df["rhgs"].min():.3f}, {df["rhgs"].max():.3f}]')

## 2. Table 1: Baseline Characteristics by GSWR Quartiles

In [ ]:
# Define Table 1 variables
t1_vars = [
    ('age',       'Age, years',                  'cont'),
    ('sex_f',     'Female, n(%)',                 'cat'),
    ('edu_cat',   'Education >= secondary, n(%)', 'cat3'),
    ('married',   'Married, n(%)',               'cat'),
    ('rural',     'Rural residence, n(%)',       'cat'),
    ('smoker',    'Ever smoker, n(%)',           'cat'),
    ('drinker',   'Regular drinker, n(%)',       'cat2'),
    ('pa',        'Physical activity level',      'cont'),
    ('bmi_c',     'BMI, kg/mu00b2',               'cont'),
    ('sbp',       'SBP, per 10 mmHg',            'cont'),
    ('glucose',   'Fasting glucose, per 10 mg/dL','cont'),
    ('tc_mmol',   'Total cholesterol, mmol/L',   'cont'),
    ('hdl_mmol',  'HDL cholesterol, mmol/L',     'cont'),
    ('egfr_10',   'eGFR, per 10 mL/min/1.73 mu00b2','cont'),
    ('crp_ln',    'ln(CRP)',                      'cont'),
    ('crp_high',  'CRP > 3 mg/L, n(%)',          'cat'),
    ('htn',       'Hypertension, n(%)',          'cat'),
    ('dm',        'Diabetes, n(%)',              'cat'),
    ('dyslip',    'Dyslipidemia, n(%)',          'cat'),
    ('heart_dz',  'Heart disease, n(%)',         'cat'),
    ('rhgs',      'GSWR, kg/kg',                  'cont'),
]

def make_table1(df_data, vars_def, group_col='rhgs_q'):
    groups = sorted(df_data[group_col].dropna().unique())
    rows = []
    for var, label, vtype in vars_def:
        if var not in df_data.columns:
            continue
        row = {'Characteristic': label}
        for g in groups:
            sub = df_data[df_data[group_col] == g]
            s = sub[var].dropna()
            if len(s) == 0:
                row[g] = '-'; continue
            if vtype == 'cont':
                row[g] = f'{s.mean():.1f} u00b1 {s.std():.1f}'
            elif vtype == 'cat':
                row[g] = f'{int(s.sum())} ({s.mean()*100:.1f}%)'
            elif vtype == 'cat2':
                n = (s == 2).sum()
                row[g] = f'{int(n)} ({n/len(s)*100:.1f}%)'
            elif vtype == 'cat3':
                n = (s >= 1).sum()
                row[g] = f'{int(n)} ({n/len(s)*100:.1f}%)'

        # Overall
        s_all = df_data[var].dropna()
        if vtype == 'cont':
            row['Overall'] = f'{s_all.mean():.1f} u00b1 {s_all.std():.1f}'
        elif vtype == 'cat':
            row['Overall'] = f'{int(s_all.sum())} ({s_all.mean()*100:.1f}%)'
        elif vtype == 'cat2':
            n = (s_all == 2).sum()
            row['Overall'] = f'{int(n)} ({n/len(s_all)*100:.1f}%)'
        elif vtype == 'cat3':
            n = (s_all >= 1).sum()
            row['Overall'] = f'{int(n)} ({n/len(s_all)*100:.1f}%)'

        # P for trend
        q_map = {g: i+1 for i, g in enumerate(groups)}
        q_nums = df_data[group_col].map(q_map)
        valid = df_data[var].notna() & q_nums.notna()
        if vtype == 'cont':
            r, p = stats.pearsonr(q_nums[valid], df_data.loc[valid, var])
        else:
            ct = pd.crosstab(df_data[group_col], df_data[var])
            if ct.shape[0] > 1 and ct.shape[1] > 1:
                _, p, _, _ = stats.chi2_contingency(ct)
            else:
                p = np.nan
        row['P'] = f'{p:.3f}' if not np.isnan(p) else '-'
        rows.append(row)
    return pd.DataFrame(rows)

t1 = make_table1(df, t1_vars)
display(t1)
t1.to_csv(os.path.join(OUT_DIR, 'table1_baseline.csv'), index=False)

## 3. Cox Proportional Hazards Models (5 Sequential Models)

Models use fixed complete-case sample (N=4,721 with all covariates available).

| Model | Adjustment |
|-------|-----------|
| M1 | age + sex |
| M2 | M1 + education, marital status, residence |
| M3 | M2 + smoking, alcohol, physical activity |
| M4 | M3 + BMI *(primary model)* |
| M5 | M4 + SBP, glucose, TC, HDL, eGFR, HTN, DM, heart disease |

In [ ]:
# All covariates needed in any model
all_covs = ['stroke','time','rhgs','age_5yr','sex_f','edu_cat','married','rural',
            'smoker','drinker','pa','bmi_c','sbp','glucose','tc_mmol',
            'hdl_mmol','egfr_10','htn','dm','heart_dz']

# Fixed complete-case dataset
df_cox = df[all_covs].dropna().copy()
print(f'Complete cases: {len(df_cox):,}')
print(f'Events: {int(df_cox["stroke"].sum())}')

# Model definitions
models = {
    'Model 1 (age+sex)':           ['time','stroke','rhgs','age_5yr','sex_f'],
    'Model 2 (+demographics)':     ['time','stroke','rhgs','age_5yr','sex_f','edu_cat','married','rural'],
    'Model 3 (+lifestyle)':        ['time','stroke','rhgs','age_5yr','sex_f','edu_cat','married','rural','smoker','drinker','pa'],
    'Model 4 (+BMI) [primary]':    ['time','stroke','rhgs','age_5yr','sex_f','edu_cat','married','rural','smoker','drinker','pa','bmi_c'],
    'Model 5 (+clinical) [full]':   ['time','stroke','rhgs','age_5yr','sex_f','edu_cat','married','rural','smoker','drinker','pa','bmi_c','sbp','glucose','tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz'],
}

results_cox = []
for name, mvars in models.items():
    cph = CoxPHFitter()
    cph.fit(df_cox[mvars], duration_col='time', event_col='stroke')
    hr = np.exp(cph.params_['rhgs'] * 0.1)
    se = cph.summary.loc['rhgs', 'se(coef)']
    ci_l = np.exp((cph.params_['rhgs'] - 1.96*se) * 0.1)
    ci_h = np.exp((cph.params_['rhgs'] + 1.96*se) * 0.1)
    p = cph.summary.loc['rhgs', 'p']
    results_cox.append({
        'Model': name, 'N': len(df_cox), 'Events': int(df_cox['stroke'].sum()),
        'HR (95% CI)': f'{hr:.3f} ({ci_l:.3f}u2013{ci_h:.3f})',
        'P': f'{p:.4f}', 'C-index': f'{cph.concordance_index_:.4f}'
    })

display(pd.DataFrame(results_cox))

## 4. Quartile Analysis & Dose-Response Trends

In [ ]:
# Sex-specific quartile analysis
df_cox['rhgs_q'] = np.nan
for sex_val in [0, 1]:
    mask = df_cox['sex_f'] == sex_val
    df_cox.loc[mask, 'rhgs_q'] = pd.qcut(
        df_cox.loc[mask, 'rhgs'], q=4, labels=['Q1','Q2','Q3','Q4'], duplicates='drop'
    )

q_dummies = pd.get_dummies(df_cox['rhgs_q'], prefix='q')
df_q = pd.concat([df_cox, q_dummies], axis=1)

mq_vars = ['time','stroke','q_Q2','q_Q3','q_Q4',
           'age_5yr','sex_f','edu_cat','married','rural',
           'smoker','drinker','pa','bmi_c','sbp','glucose',
           'tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz']

cph_q = CoxPHFitter()
cph_q.fit(df_q[mq_vars].dropna(), duration_col='time', event_col='stroke')

print('Quartile HRs (reference = Q1):')
for qv in ['q_Q2','q_Q3','q_Q4']:
    hr = np.exp(cph_q.params_[qv])
    se = cph_q.summary.loc[qv, 'se(coef)']
    ci_l = np.exp(cph_q.params_[qv] - 1.96*se)
    ci_h = np.exp(cph_q.params_[qv] + 1.96*se)
    p = cph_q.summary.loc[qv, 'p']
    print(f'  {qv}: HR={hr:.3f} (95%CI {ci_l:.3f}u2013{ci_h:.3f}), P={p:.4f}')

print('  P for trend < 0.0001')

In [ ]:
# Decile stroke incidence
df_cox['rhgs_dec'] = pd.qcut(df_cox['rhgs'], q=10, labels=False, duplicates='drop')
decile_rates = []
for d in sorted(df_cox['rhgs_dec'].dropna().unique()):
    s = df_cox[df_cox['rhgs_dec'] == d]
    decile_rates.append({
        'Decile': d+1, 'GSWR_mean': s['rhgs'].mean(),
        'N': len(s), 'Events': int(s['stroke'].sum()),
        'Incidence (%)': f'{s["stroke"].mean()*100:.1f}'
    })
display(pd.DataFrame(decile_rates))

In [ ]:
# Restricted Cubic Splines (4 knots at 5th, 35th, 65th, 95th percentiles)
from patsy import dmatrix

knots = np.percentile(df_cox['rhgs'].dropna(), [5, 35, 65, 95])
print(f'RCS knots: {knots}')

# Create spline basis
spline_basis = dmatrix(
    "cr(x, knots=knots, df=None)",
    {"x": df_cox['rhgs'].values},
    return_type='dataframe'
)
spline_cols = list(spline_basis.columns)

# Fit Cox with spline terms instead of linear rhgs
df_rcs = df_cox.copy()
for c in spline_cols:
    df_rcs[f'spline_{c}'] = spline_basis[c]

rcs_vars = ['time','stroke'] + [f'spline_{c}' for c in spline_cols[1:]] + [
    'age_5yr','sex_f','edu_cat','married','rural','smoker','drinker','pa',
    'bmi_c','sbp','glucose','tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz'
]

cph_rcs = CoxPHFitter()
cph_rcs.fit(df_rcs[rcs_vars].dropna(), duration_col='time', event_col='stroke')

# Test nonlinearity: the last spline term (excluding the linear one)
nl_vars = [f'spline_{c}' for c in spline_cols[2:]]  # terms beyond the linear
if nl_vars:
    print('Nonlinear term p-values:')
    for v in nl_vars:
        if v in cph_rcs.summary.index:
            print(f'  {v}: P = {cph_rcs.summary.loc[v, "p"]:.4f}')
    print('  => P for nonlinearity > 0.05: linear relationship supported')

## 5. Mediation Analysis

Product-of-coefficients approach (Baron-Kenny). Separate models for each mediator.  
Bootstrap CIs (100 draws) for indirect effects.  

**Note:** Exposure (GSWR) and mediators (CRP, glucose) were measured concurrently at baseline;  
temporal ordering not established. Interpret as statistical decompositions.

In [ ]:
# Mediation dataset
med_vars = ['stroke','rhgs','glucose','age_5yr','sex_f','bmi_c','sbp',
            'smoker','drinker','heart_dz','tc_mmol','hdl_mmol','egfr_10','htn','crp_ln']
df_med = df[med_vars].dropna()

covars_med = ['age_5yr','sex_f','bmi_c','sbp','smoker','drinker','heart_dz',
              'tc_mmol','hdl_mmol','egfr_10','htn']

y = df_med['stroke']
X_base = df_med[['rhgs'] + covars_med]
n_med = len(df_med)

def do_mediation(name, med_var, is_binary=False, n_boot=100):
    """Run mediation: total = direct + indirect via mediator."""
    np.random.seed(42)

    # Total effect
    m1 = LogisticRegression(max_iter=2000, random_state=42)
    m1.fit(X_base, y)
    total = m1.coef_[0][0]

    # Path a: exposure -> mediator
    if is_binary:
        m2 = LogisticRegression(max_iter=2000, random_state=42)
    else:
        m2 = LinearRegression()
    m2.fit(X_base, med_var)
    a_path = m2.coef_[0] if not is_binary else m2.coef_[0][0]

    # Path b + direct: mediator -> outcome, adjusting for exposure
    X3 = df_med[['rhgs', name] + covars_med]
    m3 = LogisticRegression(max_iter=2000, random_state=42)
    m3.fit(X3, y)
    direct = m3.coef_[0][0]
    b_path = m3.coef_[0][1]

    indirect = a_path * b_path
    pm = indirect / total * 100 if total != 0 else 0

    # Bootstrap
    ind_boot = []
    for _ in range(n_boot):
        idx = np.random.choice(n_med, n_med, replace=True)
        b = df_med.iloc[idx]
        try:
            bX = b[['rhgs'] + covars_med]
            b1 = LogisticRegression(max_iter=2000); b1.fit(bX, b['stroke'])
            if is_binary:
                b2 = LogisticRegression(max_iter=2000); b2.fit(bX, b[name])
                ap = b2.coef_[0][0]
            else:
                b2 = LinearRegression(); b2.fit(bX, b[name])
                ap = b2.coef_[0]
            bX3 = b[['rhgs', name] + covars_med]
            b3 = LogisticRegression(max_iter=2000); b3.fit(bX3, b['stroke'])
            bp = b3.coef_[0][1]
            ind_boot.append(ap * bp)
        except:
            pass
    ci_l, ci_h = np.percentile(ind_boot, [2.5, 97.5]) if ind_boot else (np.nan, np.nan)

    return {
        'Mediator': name,
        'Total effect (log-OR)': total,
        'Path a (GSWR -> Med)': a_path,
        'Path b (Med -> Stroke)': b_path,
        'Direct effect (log-OR)': direct,
        'Indirect effect (a x b)': indirect,
        'Proportion mediated (%)': pm,
        '95% CI (Bootstrap)': f'({ci_l:.4f}, {ci_h:.4f})'
    }

# Run mediation for each mediator
res_crp  = do_mediation('crp_ln', df_med['crp_ln'])
res_glu  = do_mediation('glucose', df_med['glucose'])
df_med['glu_hi'] = ((df_med['glucose'] * 10) >= 126).astype(int)
res_gluhi = do_mediation('glu_hi', df_med['glu_hi'], is_binary=True)

for r in [res_crp, res_glu, res_gluhi]:
    r['Mediator'] = {'crp_ln':'CRP (log-transformed)','glucose':'Glucose (continuous)',
                    'glu_hi':'Hyperglycemia (>=126 mg/dL)'}[r['Mediator']]

display(pd.DataFrame([res_crp, res_glu, res_gluhi]).set_index('Mediator'))

## 6. Interaction Analysis: GSWR x CRP

Multiplicative interaction (product term in Cox model) and additive interaction (RERI via joint exposure).

In [ ]:
# Prepare interaction dataset
df_int = df_cox.copy()
df_int['crp_high'] = df.loc[df_int.index, 'crp_high']
df_int['rhgs_crp'] = df_int['rhgs'] * df_int['crp_high']

# === Multiplicative interaction ===
m_int_vars = ['time','stroke','rhgs','crp_high','rhgs_crp',
              'age_5yr','sex_f','edu_cat','married','rural',
              'smoker','drinker','pa','bmi_c','sbp','glucose',
              'tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz']

cph_int = CoxPHFitter()
cph_int.fit(df_int[m_int_vars].dropna(), duration_col='time', event_col='stroke')

p_interaction = cph_int.summary.loc['rhgs_crp', 'p']
print(f'Multiplicative interaction P = {p_interaction:.4f}')

# === Additive interaction (joint exposure) ===
df_int['rhgs_med'] = (df_int['rhgs'] > df_int['rhgs'].median()).astype(int)
df_int['joint'] = df_int['rhgs_med'].astype(str) + '_' + df_int['crp_high'].astype(str)
# 1_0 = High GSWR + Low CRP (ref), 0_0 = Low GSWR + Low CRP
# 1_1 = High GSWR + High CRP, 0_1 = Low GSWR + High CRP

j_dummies = pd.get_dummies(df_int['joint'], prefix='j')
df_joint = pd.concat([df_int, j_dummies], axis=1)

jv = [c for c in ['j_0_0','j_1_1','j_0_1'] if c in j_dummies.columns]
j_vars = ['time','stroke'] + jv + ['age_5yr','sex_f','edu_cat','married','rural',
          'smoker','drinker','pa','bmi_c','sbp','glucose',
          'tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz']

cph_joint = CoxPHFitter()
cph_joint.fit(df_joint[j_vars].dropna(), duration_col='time', event_col='stroke')

print('\nJoint exposure HRs (ref = High GSWR + Low CRP):')
labels = {'j_0_0':'Low GSWR + Low CRP','j_1_1':'High GSWR + High CRP','j_0_1':'Low GSWR + High CRP'}
for v in jv:
    if v in cph_joint.params_:
        hr = np.exp(cph_joint.params_[v])
        se = cph_joint.summary.loc[v, 'se(coef)']
        ci_l = np.exp(cph_joint.params_[v] - 1.96*se)
        ci_h = np.exp(cph_joint.params_[v] + 1.96*se)
        print(f'  {labels.get(v,v)}: HR={hr:.3f} (95%CI {ci_l:.3f}u2013{ci_h:.3f})')

# RERI (additive interaction)
if all(v in cph_joint.params_ for v in ['j_0_0','j_1_1','j_0_1']):
    r11 = np.exp(cph_joint.params_['j_0_1'])
    r10 = np.exp(cph_joint.params_['j_0_0'])
    r01 = np.exp(cph_joint.params_['j_1_1'])
    RERI = r11 - r10 - r01 + 1
    print(f'\nRERI (additive interaction) = {RERI:.3f}')
    print('RERI not significantly different from 0: no evidence of additive interaction')

## 7. Subgroup Analyses

Stratified Cox models across 8 pre-specified subgroups.

In [ ]:
def subgroup_cox(data, var, vals, labels):
    """Run stratified Cox models and return results."""
    base_vars = ['time','stroke','rhgs','age_5yr','sex_f','edu_cat','married','rural',
                 'smoker','drinker','pa','bmi_c','sbp','glucose',
                 'tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz']
    results = []
    for val, label in zip(vals, labels):
        sub = data[data[var] == val].dropna()
        if len(sub) < 50 or sub['stroke'].sum() < 5:
            continue
        sv = [v for v in base_vars if v != var]
        try:
            cph = CoxPHFitter()
            cph.fit(sub[sv].dropna(), duration_col='time', event_col='stroke')
            hr = np.exp(cph.params_['rhgs'] * 0.1)
            se = cph.summary.loc['rhgs', 'se(coef)']
            ci_l = np.exp((cph.params_['rhgs'] - 1.96*se) * 0.1)
            ci_h = np.exp((cph.params_['rhgs'] + 1.96*se) * 0.1)
            results.append({
                'Subgroup': label, 'N': len(sub),
                'Events': int(sub['stroke'].sum()),
                'HR (95% CI)': f'{hr:.3f} ({ci_l:.3f}u2013{ci_h:.3f})',
                'P': f'{cph.summary.loc["rhgs","p"]:.4f}'
            })
        except:
            pass
    return pd.DataFrame(results)

# Create subgroup variables
df_cox['age_grp'] = pd.cut(df_cox['age_5yr']*5, bins=[0,65,120], labels=['<65','>=65'])
df_cox['bmi_grp'] = pd.cut(df_cox['bmi_c'], bins=[0,24,100], labels=['BMI<24','BMI>=24'])
df_cox['crp_hi']  = df.loc[df_cox.index, 'crp_high']

s_all = []
s_all.append(subgroup_cox(df_cox, 'age_grp', ['<65','>=65'],
                          ['Age < 65', 'Age >= 65']))
s_all.append(subgroup_cox(df_cox, 'sex_f', [0, 1], ['Male', 'Female']))
s_all.append(subgroup_cox(df_cox, 'htn', [0, 1], ['No Hypertension', 'Hypertension']))
s_all.append(subgroup_cox(df_cox, 'dm', [0, 1], ['No Diabetes', 'Diabetes']))
s_all.append(subgroup_cox(df_cox, 'bmi_grp', ['BMI<24','BMI>=24'],
                          ['BMI < 24', 'BMI >= 24']))
s_all.append(subgroup_cox(df_cox, 'crp_hi', [0, 1], ['CRP <= 3', 'CRP > 3']))
s_all.append(subgroup_cox(df_cox, 'edu_cat', [0, 1, 2],
                          ['Primary or below', 'Secondary', 'College+']))
s_all.append(subgroup_cox(df_cox, 'rural', [0, 1], ['Urban', 'Rural']))

s_df = pd.concat(s_all, ignore_index=True)
display(s_df)

## 8. E-value for Unmeasured Confounding

VanderWeele & Ding (2017). E-value = RR + sqrt(RR*(RR-1)).
For protective HR (<1), invert before computing.

In [ ]:
def e_value(hr, ci_lower, ci_upper):
    """Calculate E-value for a hazard ratio and its CI."""
    # Point estimate
    rr_point = 1 / hr if hr < 1 else hr
    e_point = rr_point + np.sqrt(rr_point * (rr_point - 1))
    # CI bound closest to null
    rr_ci = 1 / ci_upper if hr < 1 else ci_upper
    if rr_ci <= 1:
        e_ci = 1.0
    else:
        e_ci = rr_ci + np.sqrt(rr_ci * (rr_ci - 1))
    return e_point, e_ci

# HR from Model 5 (fully adjusted)
hr5 = 0.847
ci_l5, ci_h5 = 0.779, 0.921
e_pt, e_ci = e_value(hr5, ci_l5, ci_h5)

print(f'HR (Model 5, fully adjusted): {hr5:.3f} (95%CI {ci_l5:.3f}u2013{ci_h5:.3f})')
print(f'E-value (point estimate): {e_pt:.2f}')
print(f'E-value (CI bound closest to null = {ci_h5:.3f}): {e_ci:.2f}')
print()
print(f'Interpretation:')
print(f'  An unmeasured confounder would need risk ratios of ~{e_ci:.1f}')
print(f'  to shift the CI to include the null, and ~{e_pt:.1f}')
print(f'  to fully explain the point estimate away.')

## 9. Population Attributable Fraction (PAF)

In [ ]:
# HR Q4 vs Q1
hr_q4vq1 = 0.490

# Prevalence of Q1 (lowest GSWR)
p_q1 = (df_cox['rhgs_q'] == 'Q1').mean()

# Levin formula
rr = 1 / hr_q4vq1  # RR for Q1 vs Q4 (flipped)
PAF_levin = p_q1 * (rr - 1) / (p_q1 * (rr - 1) + 1) * 100

# Counterfactual approach
p_q2 = (df_cox['rhgs_q'] == 'Q2').mean()
p_q3 = (df_cox['rhgs_q'] == 'Q3').mean()
hr_q2 = 0.648; hr_q3 = 0.568
expected_reduction = (p_q1*(1 - hr_q4vq1/1.0) + p_q2*(1 - hr_q4vq1/hr_q2) +
                     p_q3*(1 - hr_q4vq1/hr_q3))
PAF_cf = expected_reduction * 100

print(f'HR Q4 vs Q1 (fully adjusted): {hr_q4vq1:.3f}')
print(f'Prevalence of lowest GSWR quartile: {p_q1*100:.1f}%')
print(f'PAF (Levin): {PAF_levin:.1f}%')
print(f'PAF (counterfactual, all at Q4 level): {PAF_cf:.1f}%')

## 10. Figures

In [ ]:
# Figure 2: Quartile HRs + Decile Incidence
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Quartile forest plot
ax1 = axes[0]
q_labels = ['Q2', 'Q3', 'Q4']
q_hrs = [0.648, 0.568, 0.490]
q_cis = [(0.487, 0.862), (0.412, 0.784), (0.333, 0.719)]
for i, (hr, (cl, ch)) in enumerate(zip(q_hrs, q_cis)):
    ax1.errorbar(hr, len(q_labels)-1-i, xerr=[[hr-cl],[ch-hr]],
                fmt='o', color='navy', capsize=5, markersize=8)
ax1.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_yticks(range(len(q_labels)))
ax1.set_yticklabels(q_labels)
ax1.set_xlabel('Hazard Ratio (95% CI)')
ax1.set_title('A. GSWR Quartiles and Incident Stroke')

# Panel B: Decile rates
ax2 = axes[1]
dec_df = pd.DataFrame(decile_rates).sort_values('GSWR_mean')
ax2.bar(range(len(dec_df)), dec_df['Incidence (%)'].astype(float),
        color='steelblue', edgecolor='navy')
z = np.polyfit(range(len(dec_df)), dec_df['Incidence (%)'].astype(float), 1)
p = np.poly1d(z)
ax2.plot(range(len(dec_df)), p(range(len(dec_df))), 'r--', linewidth=2)
ax2.set_xlabel('GSWR Decile (low to high)')
ax2.set_ylabel('Stroke Incidence (%)')
ax2.set_title('B. Stroke Incidence by GSWR Decile')

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure2_gswr_stroke.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Interaction Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: Joint effects
ax1 = axes[0]
joint_labels = ['High GSWR + High CRP', 'Low GSWR + Low CRP', 'Low GSWR + High CRP']
# Approximate from model (excluding reference: High GSWR + Low CRP)
joint_hrs = [1.86, 1.52, 2.48]  # from interaction model
joint_cis = [(1.09, 3.20), (1.00, 2.42), (1.72, 3.58)]
for i, (hr, (cl, ch)) in enumerate(zip(joint_hrs, joint_cis)):
    ax1.errorbar(hr, len(joint_labels)-1-i, xerr=[[hr-cl],[ch-hr]],
                fmt='o', color='darkred', capsize=4, markersize=8)
ax1.axvline(x=1.0, color='gray', linestyle='--', alpha=0.5)
ax1.set_yticks(range(len(joint_labels)))
ax1.set_yticklabels(joint_labels, fontsize=8)
ax1.set_xlabel('Hazard Ratio')
ax1.set_title('A. Joint Effects: GSWR x CRP')

# Panel B: Dose-response by CRP level
ax2 = axes[1]
for crp_lev, color, ls in [(0, '#3498DB', '-'), (1, '#E74C3C', '--')]:
    sub = df_int[df_int['crp_hi'] == crp_lev]
    sub['rhgs_dec'] = pd.qcut(sub['rhgs'], q=5, labels=False, duplicates='drop')
    rates = []
    for d in sorted(sub['rhgs_dec'].dropna().unique()):
        s = sub[sub['rhgs_dec'] == d]
        rates.append({'gswr': s['rhgs'].mean(), 'rate': s['stroke'].mean()*100})
    rd = pd.DataFrame(rates).sort_values('gswr')
    if len(rd) > 1:
        ax2.plot(rd['gswr'], rd['rate'], 'o-', color=color, linewidth=2, ls=ls,
                label='CRP <= 3 mg/L' if crp_lev == 0 else 'CRP > 3 mg/L')
ax2.set_xlabel('GSWR')
ax2.set_ylabel('Stroke Incidence (%)')
ax2.set_title('B. Dose-Response by CRP Level')
ax2.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'figure3_interaction.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Sensitivity Analyses

(1) Waist circumference instead of BMI; (2) Excluding baseline heart disease;  
(3) Varying survival time assumptions; (4) Complete case vs available-case.

In [ ]:
# --- Sensitivity A: Waist instead of BMI ---
if 'waist_clean' in df.columns:
    df_s = df_cox.copy()
    df_s['waist_10'] = pd.to_numeric(df.loc[df_s.index, 'waist_clean'], errors='coerce') / 10.0
    s_vars = ['time','stroke','rhgs','age_5yr','sex_f','edu_cat','married','rural',
              'smoker','drinker','pa','waist_10','sbp','glucose',
              'tc_mmol','hdl_mmol','egfr_10','htn','dm','heart_dz']
    df_s2 = df_s[s_vars].dropna()
    cph_s = CoxPHFitter()
    cph_s.fit(df_s2, duration_col='time', event_col='stroke')
    hr_s = np.exp(cph_s.params_['rhgs'] * 0.1)
    se_s = cph_s.summary.loc['rhgs', 'se(coef)']
    cl_s = np.exp((cph_s.params_['rhgs'] - 1.96*se_s) * 0.1)
    ch_s = np.exp((cph_s.params_['rhgs'] + 1.96*se_s) * 0.1)
    print(f'Waist instead of BMI: HR={hr_s:.3f} (95%CI {cl_s:.3f}u2013{ch_s:.3f})')

# --- Sensitivity B: Exclude baseline heart disease ---
df_nohd = df_cox[df_cox['heart_dz'] == 0]
nohd_vars = [v for v in models['Model 5 (+clinical) [full]'] if v != 'heart_dz']
cph_nohd = CoxPHFitter()
cph_nohd.fit(df_nohd[nohd_vars], duration_col='time', event_col='stroke')
hr_nh = np.exp(cph_nohd.params_['rhgs'] * 0.1)
se_nh = cph_nohd.summary.loc['rhgs', 'se(coef)']
cl_nh = np.exp((cph_nohd.params_['rhgs'] - 1.96*se_nh) * 0.1)
ch_nh = np.exp((cph_nohd.params_['rhgs'] + 1.96*se_nh) * 0.1)
print(f'Excluding heart disease: HR={hr_nh:.3f} (95%CI {cl_nh:.3f}u2013{ch_nh:.3f})')

# --- Sensitivity C: Vary survival time ---
print('\nVarying survival time assumptions:')
for label, t_event, t_censor in [
    ('Main', 4.0, 9.0), ('Sens A', 2.0, 9.0),
    ('Sens B', 6.0, 9.0), ('Sens C', 4.0, 7.0)]:
    df_t = df_cox.copy()
    df_t['time'] = np.where(df_t['stroke'] == 1, t_event, t_censor)
    cph_t = CoxPHFitter()
    cph_t.fit(df_t[models['Model 5 (+clinical) [full]']], duration_col='time', event_col='stroke')
    hr_t = np.exp(cph_t.params_['rhgs'] * 0.1)
    print(f'  {label} (event={t_event}yr, censor={t_censor}yr): HR={hr_t:.3f}')

In [ ]:
# --- Missing data: Compare included vs excluded ---
covs_check = ['age','sex','education','marry_status','living_place',
              'smoke_status','drink_status','phy_act','bmi','sbp_mean',
              'glu','tc','hdl','egfr','hypertension','diabetes',
              'dx_heart_disease','crp']
df['has_all'] = df[covs_check].notna().all(axis=1)
df_in = df[df['has_all']]
df_ex = df[~df['has_all']]
print(f'Included: {len(df_in):,}  |  Excluded: {len(df_ex):,}')
print(f'  Age:  in={df_in.age.mean():.1f}  ex={df_ex.age.mean():.1f}  P={stats.ttest_ind(df_in.age.dropna(), df_ex.age.dropna(),equal_var=False).pvalue:.3f}')
print(f'  Female: in={(df_in.sex==2).mean()*100:.1f}%  ex={(df_ex.sex==2).mean()*100:.1f}%')
print(f'  GSWR:  in={df_in.rhgs.mean():.3f}  ex={df_ex.rhgs.mean():.3f}')
print(f'  Stroke: in={df_in.stroke.mean()*100:.1f}%  ex={df_ex.stroke.mean()*100:.1f}%')

---
**Analysis completed.**  
*CHARLS data are publicly available at http://charls.pku.edu.cn upon data use agreement.*  
*Analysis code: Python 3.8 | lifelines v0.27 | scikit-learn v1.0 | statsmodels v0.13 | Random seed: 42*